# Task B -- full-data MuRIL-large capacity run

This notebook tests a larger encoder against the confirmed Task B recipe. It uses
MuRIL-large with conservative memory settings, one-layer reinitialization, R-Drop,
and five full-data seeds. FGM is disabled because its embedding-table copy does not
fit reliably with the larger model.

| setting | value |
|---|---|
| encoder | `google/muril-large-cased` |
| TAPT | Task B train + permitted OffensEval text, no holdout |
| classifier | all 3,159 Task B rows, no deduplication |
| reinitialization | one final encoder layer |
| R-Drop | 0.5 |
| FGM | disabled for memory |
| learning rate | `1e-5` |
| batch | 4 with 4-step accumulation, effective 16 |
| epochs | 6 |
| seeds | 42, 43, 44, 45, 46 |

There is no honest local F1 for this full-data fit. The output ZIP is the artifact
to submit to CodaBench. Expected runtime is approximately 5--8 hours on a T4 x2 or
P100.

In [ ]:
import os, pathlib, re, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build a MuRIL-large TAPT checkpoint

The base MuRIL TAPT checkpoint cannot be reused because its encoder is a different
size. TAPT is therefore repeated with the same permitted Task B and OffensEval text,
but initialized from `google/muril-large-cased`.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-muril-large"
TAPT_LOG = "artifacts/logs/tapt_muril_large.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing large TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--model", "google/muril-large-cased",
         "--corpus", "data/raw/multiclass_train.csv",
                    "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--epochs", "8", "--bs", "2", "--grad-accum", "8",
         "--eval-bs", "4", "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "large TAPT checkpoint was not written"
print("large TAPT checkpoint ready:", TAPT_OUT)

## 2. Train five full-data MuRIL-large models

Each seed sees all 3,159 labelled rows. The only capacity change relative to the
current recipe is the large encoder plus the memory-safe learning-rate, batch and
FGM settings.

In [ ]:
TAG = "b_muril_large_full"
TRAIN_LOG = pathlib.Path("artifacts/logs") / f"{TAG}.log"
RUN_DIR = pathlib.Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG, "--model", TAPT_OUT,
     "--folds", "1", "--no-dedupe",
     "--reinit-layers", "1", "--rdrop", "0.5", "--no-fgm",
     "--lr", "1e-5", "--bs", "4", "--grad-accum", "4",
     "--eval-bs", "16", "--aux-weight", "0",
     "--seeds", "42", "43", "44", "45", "46",
     "--epochs", "6"], log=str(TRAIN_LOG))
log_text = TRAIN_LOG.read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", log_text)
assert [seed for seed, _ in fits] == ["42", "43", "44", "45", "46"], fits
assert all(rows == "3159" for _, rows in fits), fits
assert "reinit=1" in log_text and "fgm=False" in log_text
assert (RUN_DIR / "test_probs.npy").exists(), "validation probabilities were not written"
assert (RUN_DIR / "predictions.csv").exists(), "predictions were not written"
print("five MuRIL-large full-data fits completed:", RUN_DIR)

## 3. Package the CodaBench submission

In [ ]:
import pandas as pd
PRED = RUN_DIR / "predictions.csv"
ZIP = pathlib.Path("/kaggle/working/b_muril_large_full.zip")
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "b", "--pred", str(PRED), "--out", str(ZIP)])
assert ZIP.exists(), "submission ZIP was not written"
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
pred = pd.read_csv(PRED)
assert list(pred.columns) == ["id", "label"]
assert len(pred) == 395 and pred["id"].is_unique
print("READY TO UPLOAD:", ZIP)

## 4. Preserve reproducibility files

In [ ]:
OUT = pathlib.Path("/kaggle/working/muril_large_full_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for source in [ZIP, PRED, RUN_DIR / "test_probs.npy", TRAIN_LOG, pathlib.Path(TAPT_LOG)]:
    if source.exists():
        shutil.copy2(source, OUT / source.name)
print("download:", OUT)
print("files:", sorted(x.name for x in OUT.iterdir()))